
OG GPT architecture for reference.

1) tokenization,
tokenized text enters the model
|
v
---
2) token embedding layer: (this is the first step of our model - makes embeddings from text)

3) positional embeddding layer: (projects positional encoding onto token embeddings to get an understanding of whole sentence)

4) dropout: (dropout is a method to combat overfitting where nodes in a neural network are randomly dropped (removed) from the network to introduce sparsity. This was found to help the performance of neural networks and is very important. 

---
og calls for 48 of these blocks stacked 
---

5) layerNorm 1: this restricts the scale of the layer to a constant shape. We have a lot of matrix multiplication and layer norms keep shapes consistent and playing well together. 

6) masked multi head attention - 25 heads per block

7) dropout: same as above

8) layer norm 2

9) feed forward neural network: Like our paper, this is a standard feed forward nn whose hidden layer dimensions are 6400.
---

9a) Linear layer (affine transformation)

9b) GELU activation: 

9c) Linear Layer (affine transformation)

---
---

10) dropout

11) final layer norm

12) linear output



-forward pass they use attention to predict next token


---


For our model to keep it trainable we will use 5 blocks each with 8 heads per block. The FFN hidden dimension will be 512 (4 x D_model) the d model will be 128..
context lenght is 64-128

1 million params.

vocab is 65-100


A way less gnarly idea would be to implement a transformer with just the encoder (no decoder) and a single attention block in numpy. 

This looks conceptually similar from the above one minus the many stacking layers and last part

### What im actually making (frfr)
---

Im making a micro gpt with its own proprietary rust tensor library called clem

This is a 2 block (each with a single head of attention) transformer that has an encoder, ff neural network and predicts the next token. For inference, beam search would be cool to implement but thats really ambitious so i'll probably just try and keep it simple stupid.

Going to try and train on CPU so i'm going to do my best to write some rust that's callable via python in order to minimize any extra training overhead. 

In [3]:
import clem
import math

MAX_LENGTH = 256
BATCH_SIZE = 32 # lets be nice to our cpu.

class Tokenizer:
    def __init__(self, batches: list[list[str]]):
        """
        Each batch is a list of split sentences from paragraphs. 
        TODO: Consider writing text splitting module in rust and calling from 
        
        """
        self.idx_to_token = {}
        self.token_to_idx = {}
        self.MAX_LENGTH = MAX_LENGTH
        self.BATCH_SIZE = BATCH_SIZE
        # End of text is the core special token we use.
        # In the core gpt 2 this was used for padding, unknown tokens, end of sentence, etc.
        # newer models use more sophisticated means of tokenization (with bytes) but we are keeping things stupid simple.
        self.special_token = '<|endoftext|>'
        unique_tokens = set()
        for batch in batches:
            for sentence in batch:
                for token in sentence.split(' '):
                    if token:  # skip empty strings from "word1..word2" or trailing spaces
                        unique_tokens.add(token)

        
        sorted_tokens = sorted(list(unique_tokens))
        
        idx_to_token = {index: token for index, token in enumerate(sorted_tokens)}
        token_to_idx = {token: index for index, token in enumerate(sorted_tokens)}
        
        self.idx_to_token = idx_to_token
        self.token_to_idx = token_to_idx
        self.vocab_size = len(unique_tokens) + 1
        self.idx_to_token[len(unique_tokens)] = self.special_token
        self.token_to_idx[self.special_token] = len(unique_tokens)
        
    def decode(self, index):
        """
        Take a token, if it doesnt exist return special in leu but output a response
        """
        return self.idx_to_token.get(index, self.special_token)
    
    def encode(self, token):
        """
        Inverse of above!
        """
        if token not in self.token_to_idx:
            return self.token_to_idx[self.special_token]
        return self.token_to_idx[token]
    
    def input_query_to_embedding_with_padding(self, sentence):
        """
        Used when training!
        """
        tokens = sentence.split(' ')
        
        tokens = tokens[0:self.MAX_LENGTH]
        
        if len(tokens) < self.MAX_LENGTH:
            tokens.extend([self.special_token for i in range(self.MAX_LENGTH - len(tokens))])
        
        vector = [self.encode(token) for token in tokens]
        
        return vector

    def stack_token_idx_into_tensor(self, sentence_batch:list[str]):
        print(len(sentence_batch), 'sentence batch ;en')
        padded_embeddings = [self.input_query_to_embedding_with_padding(sent) for sent in sentence_batch]
        print(len(padded_embeddings), 'padded embedding len')
        # Clem is our hand rolled torch alternative for training, the dtype here defaults to float 32.
        tensor = clem.tensor((padded_embeddings, self.BATCH_SIZE))
        print(tensor.shape, 'tensor shape')
        assert tensor.shape == (32, 256)
        return tensor
        

In [4]:
import os
# Drop conda so maturin uses the clem venv only
os.environ.pop("CONDA_PREFIX", None)
os.environ.pop("CONDA_DEFAULT_ENV", None)
os.environ.pop("CONDA_PROMPT_MODIFIER", None)
os.environ.pop("CONDA_EXE", None)
os.environ.pop("CONDA_PYTHON_EXE", None)
!cd /Users/mykalmyb/Desktop/micro-gpt/clem && maturin develop


%pip install maturin
!cd /Users/mykalmyb/Desktop/micro-gpt/clem && maturin develop --features extension-module

🐍 Found CPython 3.11 at /Users/mykalmyb/Desktop/micro-gpt/clem/bin/python
🔗 Found pyo3 bindings
📡 Using build options features from pyproject.toml
Ignoring pytest: markers 'extra == "dev"' don't match your environment
Ignoring numpy: markers 'extra == "dev"' don't match your environment
   Compiling clem v0.1.0 (/Users/mykalmyb/Desktop/micro-gpt/clem)
 --> src/creation.rs:7:21
  |
7 | use crate::tensor::{numel, Tensor};
  |                     ^^^^^
  |
  = note: `#[warn(unused_imports)]` on by default

 --> src/indexing.rs:4:36
  |
4 | use crate::tensor::{numel, Tensor, TensorCore};
  |                                    ^^^^^^^^^^

   --> src/tensor.rs:330:9
    |
330 |     use super::*;
    |         ^^^^^^^^

  --> src/indexing.rs:54:27
   |
54 |             return Ok(out.into_py(py));
   |                           ^^^^^^^
   |
   = note: `#[warn(deprecated)]` on by default

  --> src/indexing.rs:58:27
   |
58 |             return Ok(out.into_py(py));
   |                         

In [3]:
import clem
dir(clem)
help(clem.Tensor)


Help on class Tensor in module builtins:

class Tensor(object)
 |  Methods defined here:
 |  
 |  __add__(self, value, /)
 |      Return self+value.
 |  
 |  __delitem__(self, key, /)
 |      Delete self[key].
 |  
 |  __getitem__(self, key, /)
 |      Return self[key].
 |  
 |  __iter__(self, /)
 |      Implement iter(self).
 |  
 |  __len__(self, /)
 |      Return len(self).
 |  
 |  __matmul__(self, value, /)
 |      Return self@value.
 |  
 |  __mul__(self, value, /)
 |      Return self*value.
 |  
 |  __neg__(self, /)
 |      -self
 |  
 |  __radd__(self, value, /)
 |      Return value+self.
 |  
 |  __repr__(self, /)
 |      Return repr(self).
 |  
 |  __rmatmul__(self, value, /)
 |      Return value@self.
 |  
 |  __rmul__(self, value, /)
 |      Return value*self.
 |  
 |  __rsub__(self, value, /)
 |      Return value-self.
 |  
 |  __rtruediv__(self, value, /)
 |      Return value/self.
 |  
 |  __setitem__(self, key, value, /)
 |      Set self[key] to value.
 |  
 |  __sub__(

In [5]:
class Encoder:
    """
    Map the tokenized text we are training on into a vocab space which is an indexed lookup table 
    comprised of [index:int]: [value: word]
    Remember our tokenizer takes a sentence and converts it to a vector of integers which correspond to the token in our vocabulary.
    
    Next we project a positional encoding onto our sentence encodings. 
    
    we need d_model - which is our hidden dimension size. 
    for 2 blocks each containing one attention layer, we will pick an arbitray d_model that satisifies the constraints with respect to num heads
    d-model - 256
    n heads - 2
    d_head = d-model / n heads = 128

    Vocab size is derived from the tokenizer as an attribute defined in the constructor
    """
    def __init__(
        self, 
        tokenizer:Tokenizer,
        vocab_size:int, 
        num_heads:int=2,
        d_model:int = 128,
        max_seq_len:int = 256,
        max_batch_size:int=32
    ):
        self.tokenizer = tokenizer
        self.vocab_size = tokenizer.vocab_size
        assert self.vocab_size is not None

        self.num_heads = num_heads
        self.d_model = d_model # hidden dimmensions of our model.
        # initialize random embeddings - these become learned later so randn to start is normal.
        self.embedding_table = clem.randn(self.vocab_size, self.d_model)
        self.max_seq_len = max_seq_len # max seq length of our model - 256.
        self.max_batch_size = max_batch_size # 32 for CPU friendly ness. 
        self.d_head = self.d_model / self.num_heads 
        assert self.d_head is not None
    
    def create_embeddings_from_token_idxs(self, token_idx_tensor:clem.Tensor) -> clem.Tensor:
        # Convert our indices into our randomly initialized embeddings via embedding table. 
        return self.embedding_table[token_idx_tensor]

    def positionally_encode_embeddings(self, embeddings:clem.Tensor):
        # re-scale embeddings through element wise multiplication by the square root of d_model.
        # When we project the positional encoding of sentences onto our token embeddings, we are 
        # Compressing information into a singular tensor. This has the capacity to drown out the initial meaning of each token,
        # with all the new postiional information we are projecting. 

        # This first step lifts up the semantic meaning of our token embeddings to prevent their meaning from being lost when
        # positional embeddings are later projected onto them. 
        e_prime = embeddings * math.sqrt(self.d_model)
        # Start at 0 - go till d_model spaced by 2.
        i = clem.arange(0, self.d_model, 2)
        # e'{j^2} element wise exponential multiplication by -i / self.d_model * log (tensor)
        # POS is a column vector of shape (max_seq_len, 1) so it can broadcast against div_term 
        # Broadcasting is when you have two tensors with different shapes but it is still possible to 
        # multiply them together. 
        # e.g.: 
        pos = clem.arange(self.max_seq_len, dtype='float32').reshape(self.max_seq_len, 1)

        # this gives us our even indices we can use while filling our positional encoding with data.
        div_term =  clem.exp(-i / self.d_model * clem.log(clem.tensor(10000.0)))
        
        positional_encoding = clem.zeros(self.max_seq_len, self.d_model)
        positional_encoding[:, 0::2] = clem.sin(pos * div_term)
        positional_encoding[:, 1::2] = clem.cos(pos * div_term)
        
        sliced_pos = positional_encoding[:embeddings.shape[1], :]
        return sliced_pos + e_prime
        
        

In [6]:
class MultiHeadAttention:
    def __init__(
        self,
        tokenizer: Tokenizer,
        d_model: int,
        num_heads: int = 2,
    ):
        self.tokenizer = tokenizer
        self.padding_idx = tokenizer.token_to_idx[tokenizer.special_token]
        self.d_model = d_model
        self.d_head = d_model // num_heads
        self.num_heads = num_heads
        self.qkv_proj = clem.randn(d_model, 3 * d_model)
        self.out_proj = clem.randn(d_model, d_model)

    def apply_padding_mask(self, scores, idx_tensor, batch, seq):
        # scores: (B, H, S, S) — mask keys j where idx_tensor[b, j] is padding
        is_pad = clem.eq(idx_tensor, float(self.padding_idx))
        pad_mask = is_pad.reshape(batch, 1, 1, seq)
        return scores.masked_fill(pad_mask, -1e9)

    def apply_causal_mask(self, scores, seq):
        row = clem.arange(seq).reshape(seq, 1)
        col = clem.arange(seq).reshape(1, seq)
        future = clem.gt(col, row).reshape(1, 1, seq, seq)
        return scores.masked_fill(future, -1e9)

    def split_heads(self, t, batch, seq):
        t = t.reshape(batch, seq, self.num_heads, self.d_head)
        t = t.transpose(1, 2)
        return t.reshape(batch * self.num_heads, seq, self.d_head)

    def merge_heads(self, t, batch, seq):
        t = t.reshape(batch, self.num_heads, seq, self.d_head)
        t = t.transpose(1, 2)
        return t.reshape(batch, seq, self.d_model)

    def forward(self, x, idx_tensor):
        batch, seq, d_model = x.shape

        # Flatten to 2D before slicing — clem has no 3D indexing (qkv[:, :, :] fails)
        x2 = x.reshape(batch * seq, d_model)
        qkv = x2.matmul(self.qkv_proj)  # (batch * seq, 3 * d_model)

        Q = qkv[:, 0:d_model].reshape(batch, seq, d_model)
        K = qkv[:, d_model:2 * d_model].reshape(batch, seq, d_model)
        V = qkv[:, 2 * d_model:3 * d_model].reshape(batch, seq, d_model)

        Q = self.split_heads(Q, batch, seq)
        K = self.split_heads(K, batch, seq)
        V = self.split_heads(V, batch, seq)

        scores = Q.matmul(K.transpose(1, 2)) / math.sqrt(self.d_head)
        scores = scores.reshape(batch, self.num_heads, seq, seq)
        scores = self.apply_padding_mask(scores, idx_tensor, batch, seq)
        scores = self.apply_causal_mask(scores, seq)
        scores = scores.reshape(batch * self.num_heads, seq, seq)

        attn = scores.softmax(-1)
        out = attn.matmul(V)

        out = self.merge_heads(out, batch, seq)
        out2 = out.reshape(batch * seq, d_model).matmul(self.out_proj)
        return out2.reshape(batch, seq, d_model)
        

In [7]:
class FeedForwardNeuralNetwork:
    """
    Takes the tensor from our singleAttentionHeadBlock and approximates what the next 
    """
    def __init__(self, d_model:int=128):
        d_model = d_model
        d_ff = 4 * d_model

        self.w1 = clem.randn(d_model, d_ff)
        self.b1 = clem.zeros(d_ff)
        self.w2 = clem.randn(d_ff, d_model)
        self.b2 = clem.zeros(d_model)


    def forward(self, x):
        batch, seq, d_model = x.shape
        flat = x.reshape(batch * seq, d_model)
        # Affine 1
        h = flat.matmul(self.w1) + self.b1
        # Activation function
        h = clem.gelu_fn(h)
        # Affine 2
        out = h.matmul(self.w2) + self.b2
        return out.reshape(batch, seq, d_model)

In [ ]:
class OutputHead:
    def __init__(self, d_model: int, vocab_size: int):
        self.vocab_size = vocab_size
        self.proj = clem.randn(d_model, vocab_size)

    def __call__(self, x: clem.Tensor) -> clem.Tensor:
        batch, seq, d_model = x.shape
        flat = x.reshape(batch * seq, d_model)
        logits = flat.matmul(self.proj)
        return logits.reshape(batch, seq, self.vocab_size)


class MicroGPT:
    """Really small - like clem"""

    def __init__(self, tokenizer, encoder, attention_heads, feed_forward_network=None):
        self.tokenizer = tokenizer
        self.encoder = encoder
        self.multi_head_attention = attention_heads
        self.feed_forward_network = feed_forward_network
        self.output_head = OutputHead(encoder.d_model, tokenizer.vocab_size)

    def forward(self, input_sentence: str):
        # Tokenize sentence, 
        # Pass through attention
        # go through feed forward network layers
        # Then finally through output head
        # then decode.
        # auto regressively.
        encoded_sentence = self.tokenizer.input_query_to_embedding_with_padding(input_sentence)
        embeddings = self.encoder.create_embeddings_from_token_idxs(encoded_sentence)
        positionally_encoded_embeddings = self.encoder.positionally_encode_embeddings(embeddings)
        



In [ ]:
%pip install pymupdf
from pathlib import Path
import pymupdf
import math
from dotenv import load_dotenv
import os

load_dotenv()

pdf = pymupdf.open(Path(os.environ.get('LOCAL_PDF_PATH')))
corpus = '\n--- PAGE BREAK ---\n'.join([page.get_text() for page in pdf])

all_sentences = corpus.split('.')
num_batches = math.ceil(len(all_sentences) / BATCH_SIZE)
batches = []

for i in range(num_batches):
    start = i * BATCH_SIZE
    end = min((i + 1) * BATCH_SIZE, len(all_sentences))
    batches.append(all_sentences[start:end])
    # Next step is passing the logits into our loss function in order for our network to learn!
    # This usually happens with cross entropy loss. 



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
tokenizer = Tokenizer(batches)
encoder = Encoder(tokenizer, vocab_size=tokenizer.vocab_size)
feed_forward_network = FeedForwardNeuralNetwork(d_model=encoder.d_model)

multi_head_attention = MultiHeadAttention(
    tokenizer=tokenizer,
    d_model=encoder.d_model,
)
# feed_forward_network = FeedForwardNeuralNetwork()
model = MicroGPT(
    tokenizer=tokenizer,
    encoder=encoder,
    attention_heads=multi_head_attention,
    feed_forward_network=feed_forward_network,
)

epochs = 1
# Training on CPU for starters.
for epoch in range(epochs):
    for sentence_batch in batches[:2]:
        if len(sentence_batch) != BATCH_SIZE:
            continue
        
        idx_tensor = model.tokenizer.stack_token_idx_into_tensor(sentence_batch)
        embeddings = model.encoder.create_embeddings_from_token_idxs(idx_tensor)
        print(embeddings, 'embeddings')
        x = model.encoder.positionally_encode_embeddings(embeddings)
        print(x, 'encoder x')
        # What you see happen as these run, the values get amplified.
        # Logits are projected through a 21k vocabulary. while our initial embeddings are the smallest.
        # Second smallest is the positionally encoded embeddings added to our initial embeddings. 


        # Note this is slow af because of the matrix multiplication! Clem - our vibe coded rust framework doens't use 
        # BLAS to multiply matrices which slows down multi head attention substantially. 
        x = model.multi_head_attention.forward(x, idx_tensor)
        print(x, 'multi head attention forward')
        x = model.feed_forward_network.forward(x)
        logits = model.output_head(x)
        print(logits.shape)
        batch, seq, vocab = logits.shape
        padding_idx = model.tokenizer.token_to_idx[model.tokenizer.special_token]
        flat_logits = logits.reshape(batch * seq, vocab)
        padding_idx = model.tokenizer.token_to_idx[model.tokenizer.special_token]
        print(pred_logits, next_tokens)

        loss = None
        for b in range(batch):
                pred = flat_logits[b * seq : b * seq + (seq - 1), :]   # rows, not cols
                targ = idx_tensor[b, 1:]
                bl = clem.cross_entropy(pred, targ, ignore_index=padding_idx)
                loss = bl if loss is None else loss + bl
        
        loss = loss / batch
        for p in model.parameters():
            p.zero_grad()

        loss.backward()

        lr
        # Need to update weights here based on loss!

        print(logits, 'logits')

32 sentence batch ;en
32 padded embedding len
(32, 256) tensor shape
Tensor(shape=[32, 256, 128], data=[0.8505, -0.5382, 0.7695, 1.4332, -0.1107, 0.5873...]) embeddings
Tensor(shape=[32, 256, 128], data=[9.6229, -5.0890, 8.7055, 17.2153, -1.2521, 7.6450...]) encoder x
Tensor(shape=[32, 256, 128], data=[-145.9944, -822.7003, 903.8994, -1268.6965, 394.4017, 1179.0449...]) multi head attention forward
(32, 256, 21042)


ValueError: cannot reshape [8192, 21041] to [8160, 21042]

In [ ]:
test_query = 'im hardly a good cook'

prediction = model.forward(test_query)


In [35]:
model.encoder